# Assignment, transportation & network flow

Solvers in the **assignment-matrix** and **edge-flow** shapes. Each returns a `SolveResult` carrying the result `frame`, `status`, and `objective`.

In [ ]:
import pandas as pd

import ortidy

## Linear assignment

A cost matrix *is* a dataframe: assign each agent to one task at minimum total cost. Adds `assignedTo` and `cost`.

In [ ]:
costs = pd.DataFrame({"t0": [4, 1, 3], "t1": [2, 5, 2], "t2": [8, 4, 1]})
result = ortidy.assignment(costs)
print(result.status, "| total cost:", result.objective)
result.frame

## Generalized assignment (GAP)

Assign tasks to **capacity-limited** agents to maximize value; each task consumes a size on its agent.

In [ ]:
values = pd.DataFrame({"a0": [10, 8, 5], "a1": [6, 9, 7]})
sizes = pd.DataFrame({"a0": [3, 4, 2], "a1": [2, 3, 4]})
result = ortidy.generalized_assignment(values, sizes, {"a0": 5, "a1": 6})
print(result.status, "| total value:", result.objective)
result.frame

## Transportation

Ship from sources to sinks at minimum cost, given supply and demand. Returns a tidy edge list of shipments.

In [ ]:
costs = pd.DataFrame({"src": ["S0", "S1"], "k0": [4, 5], "k1": [3, 2], "k2": [1, 3]})
result = ortidy.transportation(
    costs, supply={"S0": 10, "S1": 15}, demand={"k0": 8, "k1": 9, "k2": 8},
    source_id_column="src",
)
print(result.status, "| total cost:", result.objective)
result.frame[result.frame["quantity"] > 0]

## Network flow

Max flow, min-cost flow, and shortest path over an edge list — each adds a flow column.

In [ ]:
edges = pd.DataFrame({
    "from": [0, 0, 1, 2],
    "to": [1, 2, 2, 3],
    "capacity": [3, 2, 2, 4],
})
mf = ortidy.max_flow(edges, source=0, sink=3)
print("max flow:", mf.objective)
mf.frame

In [ ]:
edges = pd.DataFrame({
    "from": [0, 0, 1, 2],
    "to": [1, 2, 2, 3],
    "capacity": [10, 10, 10, 10],
    "cost": [4, 1, 1, 1],
})
supplies = pd.DataFrame({"node": [0, 3], "supply": [5, -5]})
mcf = ortidy.min_cost_flow(edges, supplies)
print("min cost:", mcf.objective)
mcf.frame

In [ ]:
edges = pd.DataFrame({
    "from": [0, 0, 1, 2],
    "to": [1, 2, 3, 3],
    "weight": [1, 4, 1, 1],
})
sp = ortidy.shortest_path(edges, source=0, sink=3)
print("path length:", sp.objective)
sp.frame[sp.frame["onPath"] == 1]

## Facility location

Open facilities (paying a setup cost) and assign each customer to one, minimizing opening + assignment cost.

In [ ]:
costs = pd.DataFrame({"f0": [1, 1, 9], "f1": [9, 9, 1]})
setup = pd.DataFrame({"facility": ["f0", "f1"], "setupCost": [2, 2]})
result = ortidy.facility_location(costs, setup)
print(result.status, "| total cost:", result.objective, "| opened:", result.metadata["opened"])
result.frame

## Set cover

Pick the cheapest collection of subsets that covers every element.

In [ ]:
subsets = pd.DataFrame({
    "subset": ["A", "B", "C"],
    "e0": [1, 0, 1], "e1": [1, 1, 0], "e2": [0, 1, 1], "e3": [0, 1, 0],
    "cost": [3, 2, 2],
})
result = ortidy.set_cover(subsets, subset_id="subset")
print(result.status, "| total cost:", result.objective)
result.frame